# nemotron-ethics-evaluation on Google Colab

This notebook provides a step-by-step guide to set up and run the `nemotron-ethics-evaluation` project on Google Colab, leveraging a GPU runtime for optimal performance with vLLM.

## Step 1: Change Runtime Type to GPU

Before proceeding, ensure your Colab runtime is set to GPU. Go to `Runtime` > `Change runtime type` and select `GPU` (e.g., T4).

**Note on VRAM:** The `nvidia/Nemotron-Content-Safety-Reasoning-4B` model with vLLM overhead may require significant VRAM. A 16 GB GPU might be tight, and you might encounter Out-Of-Memory (OOM) errors. If this happens, consider a Colab Pro subscription for larger GPUs or try reducing concurrency in vLLM (though this notebook uses default settings).



## Step 1b: Verify GPU (run this before cloning)

vLLM **requires** a visible NVIDIA GPU. Run the next cell: it must print an `nvidia-smi` summary. If it errors, open **Runtime → Change runtime type → Hardware accelerator: GPU** (T4 or better), click **Save**, then **Runtime → Restart runtime** and run from the top.


In [ ]:
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "nvidia-smi not found. Colab: Runtime → Change runtime type → GPU → Save, "
        "then Runtime → Restart runtime, then re-run this notebook from the top."
    )

r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise RuntimeError("nvidia-smi failed — no usable GPU in this session.")


## Step 2: Clone the Repository and Navigate into it

In [1]:
import os
import subprocess

REPO_URL = "https://github.com/libanmohamud-spec/nemotron-ethics-evaluation.git"
REPO_DIR = "nemotron-ethics-evaluation"

# If this cell is re-run from inside the repo, do not clone again (avoids
# /content/.../nemotron-ethics-evaluation/nemotron-ethics-evaluation/).
if os.path.isfile("requirements.txt") and os.path.isdir("config"):
    REPO_ROOT = os.path.abspath(".")
else:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    REPO_ROOT = os.path.abspath(REPO_DIR)
    os.chdir(REPO_ROOT)

os.environ["REPO_ROOT"] = REPO_ROOT
print("Working directory:", os.getcwd())

/content/nemotron-ethics-evaluation


## Step 3: Install Dependencies

This step will install all required Python packages. The first installation might take some time.

In [2]:
# Run after the clone cell (REPO_ROOT must be set).
# Order matters: CUDA PyTorch first, then requirements, then vLLM — otherwise
# PyTorch can stay CPU-only and vLLM raises "Failed to infer device type".
import os
import subprocess
import sys

repo = os.environ.get("REPO_ROOT") or os.path.abspath("nemotron-ethics-evaluation")
req = os.path.join(repo, "requirements.txt")
if not os.path.isfile(req):
    raise FileNotFoundError(f"Missing {req} — run the clone cell first.")

TORCH_CUDA = "https://download.pytorch.org/whl/cu124"
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "torch",
        "torchvision",
        "torchaudio",
        "--index-url",
        TORCH_CUDA,
    ]
)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", req])

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "torch",
        "torchvision",
        "torchaudio",
        "--index-url",
        TORCH_CUDA,
    ]
)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "vllm",
        "--extra-index-url",
        TORCH_CUDA,
    ]
)

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "WARNING: PyTorch still has no CUDA. Use Runtime → GPU, Restart runtime, "
        "then re-run from Step 1b. If needed, change TORCH_CUDA to cu121 in this cell."
    )
print("Dependencies installed.")


## Step 4: Set Up Hugging Face Token (if required)

If the `nvidia/Nemotron-Content-Safety-Reasoning-4B` model is gated and requires authentication, you'll need to provide your Hugging Face token. You can set it as a Colab secret named `HF_TOKEN`.


In [5]:
import os

# Optional: Colab Secrets → add key HF_TOKEN if the model or weights need auth
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN loaded from Colab secrets.")
except Exception as exc:
    print(
        "No HF token from Colab secrets (ok if the model loads without auth):",
        type(exc).__name__,
    )

Hugging Face token loaded from Colab secrets.


## Step 5: Start the vLLM server (background)

vLLM exposes an OpenAI-compatible API on **port 8001**. The readiness check uses **`/health`** (not `/v1/health`). Logs: **`vllm_server.log`**.

**Colab T4 (16GB):** Default cap is **`--max-model-len 4096`** (override with env `VLLM_MAX_MODEL_LEN`). The server also sets **`VLLM_USE_V1=0`** (legacy engine), which is more reliable on T4 than the v1 stack. If init still fails, try `VLLM_MAX_MODEL_LEN=2048` or a larger GPU.


In [6]:
import os
import subprocess
import sys
import time
import requests

try:
    import torch
except ImportError as exc:
    raise RuntimeError("Run the install cell first (needs torch).") from exc

if not torch.cuda.is_available():
    raise RuntimeError(
        "torch.cuda.is_available() is False. Colab: Runtime → Change runtime type → GPU, "
        "Runtime → Restart runtime, run Step 1b then clone → install → this cell."
    )

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print("vLLM will use:", torch.cuda.get_device_name(0))

vllm_server_process = None

# vLLM exposes GET /health (root app), not /v1/health
HEALTH_URLS = (
    "http://127.0.0.1:8001/health",
    "http://127.0.0.1:8001/v1/models",
)


def start_vllm_server():
    global vllm_server_process
    if vllm_server_process is not None and vllm_server_process.poll() is None:
        print("vLLM server already running.")
        return

    log_path = os.path.join(os.environ.get("REPO_ROOT", os.getcwd()), "vllm_server.log")
    log_f = open(log_path, "w", encoding="utf-8")
    print("Starting vLLM server in background; logging to:", log_path)

    # T4 16GB: keep context small; v1 engine often fails engine init on Colab — use legacy.
    max_len = os.environ.get("VLLM_MAX_MODEL_LEN", "4096")
    cmd = [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        "nvidia/Nemotron-Content-Safety-Reasoning-4B",
        "--port",
        "8001",
        "--dtype",
        "bfloat16",
        "--tensor-parallel-size",
        "1",
        "--max-model-len",
        max_len,
        "--gpu-memory-utilization",
        "0.85",
        "--enforce-eager",
    ]
    env = os.environ.copy()
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    env.setdefault("VLLM_USE_V1", "0")
    print("vLLM flags: max-model-len=", max_len, " VLLM_USE_V1=", env.get("VLLM_USE_V1"))
    vllm_server_process = subprocess.Popen(
        cmd,
        stdout=log_f,
        stderr=subprocess.STDOUT,
        env=env,
        cwd=os.environ.get("REPO_ROOT", os.getcwd()),
    )
    print(f"vLLM PID: {vllm_server_process.pid}")


def check_server_ready(timeout=1200, interval=5):
    print("Waiting for vLLM to become ready (first download can take many minutes)...")
    start = time.time()
    while time.time() - start < timeout:
        for url in HEALTH_URLS:
            try:
                r = requests.get(url, timeout=5)
                if r.status_code == 200:
                    print("Ready:", url)
                    return True
            except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
                pass
        time.sleep(interval)
    print("Timeout. Tail of", os.path.join(os.environ.get("REPO_ROOT", "."), "vllm_server.log"), ":")
    log_path = os.path.join(os.environ.get("REPO_ROOT", os.getcwd()), "vllm_server.log")
    if os.path.isfile(log_path):
        with open(log_path, "r", encoding="utf-8", errors="replace") as f:
            print(f.read()[-4000:])
    return False


start_vllm_server()
if not check_server_ready():
    print(
        "Failed to start vLLM server. Check vllm_server.log (OOM → "
        "os.environ['VLLM_MAX_MODEL_LEN']='2048' then re-run; or use "
        "a larger GPU)."
    )



Starting vLLM server in background...
vLLM server process started with PID: 4053
Waiting for vLLM server at http://localhost:8001/v1/health to be ready...
vLLM server did not become ready within the timeout period.
Failed to start vLLM server. Please check the logs (if redirected) or model access.


## Step 6: Run the evaluation scripts

With vLLM listening on `localhost:8001`, run the cells below (they use `REPO_ROOT` and `subprocess`, not shell magic). Optional: `python generate_audit_log.py` and `python summarise_results.py` from the same directory.


In [1]:
import os
import subprocess
import sys

repo = os.environ.get("REPO_ROOT", os.path.abspath("nemotron-ethics-evaluation"))
script = os.path.join(repo, "run_functional_tests.py")
if not os.path.isfile(script):
    raise FileNotFoundError(f"Expected {script!r} — run the clone cell first.")

print("Running functional tests...")
subprocess.run([sys.executable, "run_functional_tests.py"], cwd=repo, check=False)
print("Functional tests finished.")

Running functional tests...
python3: can't open file '/content/{repo_dir}/run_functional_tests.py': [Errno 2] No such file or directory
Functional tests complete.


In [8]:
import os
import subprocess
import sys

repo = os.environ.get("REPO_ROOT", os.path.abspath("nemotron-ethics-evaluation"))
script = os.path.join(repo, "run_ethical_tests.py")
if not os.path.isfile(script):
    raise FileNotFoundError(f"Expected {script!r} — run the clone cell first.")

print("Running ethical tests...")
subprocess.run([sys.executable, "run_ethical_tests.py"], cwd=repo, check=False)
print("Ethical tests finished.")

Running ethical tests...
python3: can't open file '/content/nemotron-ethics-evaluation/nemotron-ethics-evaluation/run_ethical_tests.py': [Errno 2] No such file or directory
Ethical tests complete.


## Step 7: View evidence plots (after a successful run)

`run_functional_tests.py` writes **`evidence/functional/latency_plot.png`** (and CSV/text). Placeholder figures under `evidence/functional/` and `evidence/ethical/` are replaced when you save screenshots or plots from your run. Run the next cell to display every **`.png`** under `evidence/`.


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

repo = Path(os.environ.get("REPO_ROOT", os.path.abspath("nemotron-ethics-evaluation")))
evidence = repo / "evidence"
pngs = sorted(evidence.rglob("*.png")) if evidence.is_dir() else []

if not pngs:
    print(f"No PNG files under {evidence}. Run the functional tests first (creates latency_plot.png).")
else:
    for p in pngs:
        print(p.relative_to(repo))
        display(Image(filename=str(p), width=900))


## Optional: Clean up vLLM Server (if manually stopping or restarting)

If you need to stop the vLLM server manually or restart it, you can run the following cell. This is typically not necessary unless you encounter issues or want to free up resources.

In [9]:
if 'vllm_server_process' in globals() and vllm_server_process and vllm_server_process.poll() is None:
    print("Terminating vLLM server process...")
    vllm_server_process.terminate()
    vllm_server_process.wait()
    print("vLLM server terminated.")
else:
    print("vLLM server not running or already terminated.")

vLLM server not running or already terminated.
